# Stage 04 QC — Candidate comparison, drift, interference diagnostics

**Companion to `04_calibration.ipynb`.** That notebook applies one locked-in method per
species (`CAL_METHOD_LOCKED`) and writes `calibration_coefs.json`/`04_calibrated/`. This
notebook is where the evidence behind each lock lives: every viable candidate method is
computed and compared here, side by side, on a common yardstick
(`cal.compare_candidate_coefs`) — plus drift QC across all three tank dates and the
interference diagnostics behind the Ultra321 C2H6 flag.

**Writes no output files.** Read-only against Stage 03 data; re-derives its own copy of
Sections A/B independently rather than importing from the official notebook (same
standalone-notebook precedent as `03a_align_wyo.ipynb`/`03b_align_mml.ipynb`) — the
duplication is a handful of short cells, intentional, so this notebook's ability to run
never depends on the official notebook's exact cell contents/order.

**How the three species differ here:**
- **CH4** — both `tank` and `reference` (Picarro cross-cal) candidates are viable for
  Ultra460/Ultra321/Pico017. This is the one species with a real comparison to make —
  Section C below.
- **C3H8** — only Ultra321 measures it, so there is no reference partner to cross-cal
  against. Nothing to compare — Section D just says so.
- **C2H6** — the tank has one certified point (NOAA, 1.63 ppb), far below plume levels,
  so `tank` isn't viable. Section E quantifies *how* inadequate (`assess_tank_coverage`)
  and carries the peak-matching illustration and interference diagnostics that justify
  both the `reference` lock and the Ultra321 low-confidence flag.

If new data changes any of this evidence, update `CAL_METHOD_LOCKED` in
`04_calibration.ipynb` (with a matching code change to its Section C/D) — this notebook
doesn't write that lock automatically; it's a human-reviewed decision.

In [ ]:
import json
from pathlib import Path

import importlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, TANK_DETAILS_PATH, REPO_ROOT
from src import calibration as cal
importlib.reload(cal)   # pick up edits to src/calibration.py without restarting the kernel
from src.align import load_aligned_series

print('Imports OK — calibration module reloaded')

In [ ]:
# ── Duplicated from 04_calibration.ipynb by design ──────────────────────────────
# This notebook stands alone (see intro markdown) rather than importing state from the
# official notebook, so its config is its own copy.
INSTRUMENTS = {
    'Picarro':  {'dir': 'WYO_picarro',        'subdir': ''},
    'Ultra460': {'dir': 'WYO_aerisultra460',  'subdir': 'Raw'},
    'Ultra321': {'dir': 'LANL_aerisultra321', 'subdir': 'Raw'},
    'Pico017':  {'dir': 'LANL_aerispico017',  'subdir': 'Raw'},
}
INST_COLORS = {
    'Picarro':  '#1f77b4',
    'Ultra460': '#ff7f0e',
    'Ultra321': '#2ca02c',
    'Pico017':  '#d62728',
}
CAL_DATE_CANONICAL = '20260212'

def load_full_series(inst, col):
    cfg = INSTRUMENTS[inst]
    return load_aligned_series(STAGE_03_DIR, cfg['dir'], cfg['subdir'], col)

with open(STAGE_02_DIR / 'routing_manifest.json') as _f:
    ROUTING_MANIFEST = json.load(_f)
WYO_DATES = cal.dates_for_platform(ROUTING_MANIFEST, 'WYO')

# Read-only cross-reference to what's currently locked in the official notebook, so a
# stale QC re-run visibly disagrees with the applied output if someone edits one without
# the other.
CAL_METHOD_LOCKED = {'CH4': 'tank', 'C3H8': 'tank', 'C2H6': 'reference'}
print('Currently locked in 04_calibration.ipynb (as of this QC notebook\'s last edit):', CAL_METHOD_LOCKED)

---
## A/B — Parse tank manifest + load Stage 03 data

Same as the official notebook's Sections A/B — duplicated here by design (see intro).

In [ ]:
TANK, WINDOWS_BY_DATE = cal.parse_tank_details(TANK_DETAILS_PATH)

CH4  = {inst: load_full_series(inst, 'CH4_ppm') for inst in INSTRUMENTS}
C3H8 = {'Ultra321': load_full_series('Ultra321', 'C3H8_ppm')}
C2H6 = {
    'Ultra460': load_full_series('Ultra460', 'C2H6_ppb'),
    'Pico017':  load_full_series('Pico017', 'C2H6_ppb'),
    'Ultra321': load_full_series('Ultra321', 'C2H6_ppm') * 1000.0,
}

ZOOM_START = pd.Timestamp('2026-02-05 20:00', tz='UTC')   # same window as the official notebook
ZOOM_END   = ZOOM_START + pd.Timedelta('180min')

print('Tank dates:', sorted(WINDOWS_BY_DATE))
for inst, s in CH4.items():
    print(f'{inst:10s} CH4 {len(s):>9,} pts')

---
## C — CH4: `tank` vs `reference` candidates

Both methods are viable here. **Tank**: multi-point OLS per instrument against the
Feb-12 dilution ladder (also fit on Feb 3/Feb 6, for the drift check below). **Reference**:
each Aeris unit's ambient CH4 cross-calibrated directly against Picarro's own continuous
field CH4 (`pair_series_nearest` + `fit_reference_cal(anchor='ols')`) — hundreds of
thousands of paired ambient points instead of 7 discrete tank-window means. Picarro
itself has no reference counterpart (it *is* the candidate reference), so it's
tank-only, same as the official notebook.

In [ ]:
# Tank candidate, all three dates (Feb 12 canonical + Feb 3/Feb 6 for the drift check below)
CH4_STATS = {d: cal.window_stats(CH4, w) for d, w in WINDOWS_BY_DATE.items()}
CH4_COEFS_BY_DATE = {d: {inst: cal.fit_species(CH4_STATS[d], TANK, inst, 'CH4_ppm')
                         for inst in INSTRUMENTS} for d in WINDOWS_BY_DATE}
CH4_TANK_COEFS = CH4_COEFS_BY_DATE[CAL_DATE_CANONICAL]

# Reference candidate: Picarro cross-cal, ambient overlap only (tank windows excluded)
PICARRO_DATES = sorted({d.strftime('%Y%m%d') for d in pd.DatetimeIndex(CH4['Picarro'].dropna().index).normalize().unique()})
CROSS_CAL_DATES = {
    'Ultra460': PICARRO_DATES,
    'Ultra321': cal.dates_for_platform(ROUTING_MANIFEST, 'WYO', prefix='Ultra100321'),
    'Pico017':  cal.dates_for_platform(ROUTING_MANIFEST, 'WYO', prefix='Pico100017'),
}
CH4_PICARRO_AMB = cal.restrict_series(CH4['Picarro'], PICARRO_DATES, WINDOWS_BY_DATE)

CH4_CROSS_COEFS, CH4_CROSS_PAIRS = {}, {}
for inst, dates in CROSS_CAL_DATES.items():
    amb = cal.restrict_series(CH4[inst], dates, WINDOWS_BY_DATE)
    paired = cal.pair_series_nearest(CH4_PICARRO_AMB, amb, tolerance_s=1)
    CH4_CROSS_COEFS[inst] = cal.fit_reference_cal(paired['ref'].values, paired['target'].values, anchor='ols')
    CH4_CROSS_PAIRS[inst] = paired
    c = CH4_CROSS_COEFS[inst]
    print(f'{inst}: cross-cal vs Picarro  n={c["n"]:,}  slope={c["slope"]:.4f}  intercept={c["intercept"]:+.4f}  R2={c["r2"]:.5f}')

**The comparison** — `cal.compare_candidate_coefs` applied per instrument, both
candidates scored on the same yardstick (that instrument's own Feb-12 tank-window
means): the tank fit was trained on these points; the cross-cal candidate wasn't, so
this is a genuine out-of-sample check for it, with a real caveat — cross-cal training
data is >99.9% ambient (<5 ppm); only ~0.04% of paired points exceed 10 ppm, essentially
all from a single large plume event. Expect the tank fit to win here (it's what the
plume-detection campaign needs); expect cross-cal to look tighter if you instead scored
it on its own ambient population (shown separately below, a different, much larger
population — not an apples-to-apples number against the table below).

In [ ]:
comparison_rows = []
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    candidates = {'tank': CH4_TANK_COEFS.get(inst), 'reference': CH4_CROSS_COEFS.get(inst)}
    cmp = cal.compare_candidate_coefs(candidates, CH4_STATS[CAL_DATE_CANONICAL], TANK, 'CH4_ppm', inst)
    cmp.insert(0, 'instrument', inst)
    comparison_rows.append(cmp)
CH4_COMPARISON = pd.concat(comparison_rows)
CH4_COMPARISON.round(4)

In [ ]:
# For context only (not a like-for-like comparison with the table above): cross-cal
# RMS scored on its OWN ambient training population instead of the 7 tank points.
own_pop_rows = []
for inst, paired in CH4_CROSS_PAIRS.items():
    c = CH4_CROSS_COEFS[inst]
    resid = cal.apply_linear(paired['target'], c) - paired['ref']
    own_pop_rows.append({'instrument': inst, 'cross_cal_RMS_at_own_ambient_pop': float(np.sqrt(np.mean(resid ** 2))),
                         'n': c['n']})
pd.DataFrame(own_pop_rows).set_index('instrument').round(4)

**Matched-pairs scatter** (fit + 1:1 line) for the reference candidate — the
generic `calibrate_and_check_reference` entry point a future campaign would call
directly for any reference-instrument calibration. Refits internally (deterministically
identical to `CH4_CROSS_COEFS` above); plotted on a 3,000-point subsample per instrument
so the figures stay light — the fit itself always uses the full paired set.

In [ ]:
for inst, paired in CH4_CROSS_PAIRS.items():
    cal.calibrate_and_check_reference(
        paired['ref'].values, paired['target'].values, anchor='ols',
        colors=INST_COLORS, x_label='Picarro CH4 (ppm, ambient)',
        y_label='Instrument CH4 (ppm, ambient)', target_label=inst,
        title_prefix='CH4 reference candidate vs Picarro — ',
        plot_sample_size=3000, plot_seed=0)

**Does the reference candidate's correction land on Picarro?** Reference (solid),
each target's raw (dashed) and corrected (solid), over the same `ZOOM_START`/`ZOOM_END`
window as the official notebook.

In [ ]:
cal_ch4_cross = {inst: cal.apply_linear(CH4[inst], c) for inst, c in CH4_CROSS_COEFS.items()}
_cmp_colors = {'Picarro (reference)': INST_COLORS['Picarro'],
               'Ultra460': INST_COLORS['Ultra460'], 'Ultra321': INST_COLORS['Ultra321'],
               'Pico017': INST_COLORS['Pico017']}
fig = cal.plot_raw_corrected_vs_reference(
    CH4['Picarro'], 'Picarro (reference)',
    {inst: (CH4[inst], cal_ch4_cross[inst]) for inst in CH4_CROSS_COEFS},
    _cmp_colors, ZOOM_START, ZOOM_END,
    title=f'CH4 — reference candidate vs Picarro — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC',
    y_title='CH4 (ppm)')
fig.show()

### Drift QC — does the locked Feb-12 tank calibration hold for the whole campaign?

**What "drift" means:** an instrument's response can change over time (aging optics,
temperature history, etc.), so a calibration measured on one day slowly becomes *wrong*
on later days. The official notebook calibrates everything from the **Feb 12** tank
event and applies those coefficients campaign-wide. That's only justified if the
instruments *didn't* drift between the tank events. Two complementary views below:

1. **Apply the one Feb-12 calibration to *every* tank date.** If an instrument didn't
   drift, its corrected readings land on the 1:1 line on Feb 3 and Feb 6 too — not just
   on Feb 12 (where it's guaranteed to, since that's the data it was fit on).
2. **Residual error vs. the fit-noise floor.** The RMS error left after applying the
   Feb-12 calibration to each date. The Feb-12 point is just the within-day fit noise; if
   Feb 3/Feb 6 sit no higher, there is no drift signal above the noise.

*(C3H8 was also fit at all three dates but doesn't get this same drift view here — kept
out of scope for this pass, matching the original notebook.)*

In [ ]:
_date_color = {'20260203': '#9ecae1', '20260206': '#4292c6', '20260212': '#08519c'}
fig = make_subplots(rows=2, cols=2, subplot_titles=list(INSTRUMENTS))
_seen = set()
for idx, inst in enumerate(INSTRUMENTS):
    rr, cc = idx // 2 + 1, idx % 2 + 1
    c = CH4_TANK_COEFS[inst]
    for d in sorted(WINDOWS_BY_DATE):
        sd = CH4_STATS[d]
        col = f'{inst}_mean'
        if c is None or col not in sd.columns:
            continue
        xt = sd['tank_key'].map(lambda k: TANK.get(k, {}).get('CH4_ppm')).astype(float)
        yc = cal.apply_linear(sd[col].astype(float), c)
        scol = f'{inst}_std'
        ye = (sd[scol].astype(float) / c['slope']).values if scol in sd.columns else None
        fig.add_trace(go.Scatter(x=xt, y=yc, mode='markers',
                                 marker=dict(color=_date_color.get(d, 'gray'), size=8),
                                 error_y=(dict(type='data', array=ye, visible=True, thickness=1, width=2)
                                          if ye is not None else None),
                                 name=d, legendgroup=d, showlegend=d not in _seen), row=rr, col=cc)
        _seen.add(d)
    fig.add_trace(go.Scatter(x=[0, 60], y=[0, 60], mode='lines',
                             line=dict(color='rgba(0,0,0,0.4)', dash='dash'),
                             showlegend=False), row=rr, col=cc)
fig.update_layout(
    title='CH4: single Feb-12 calibration applied to every tank date — on 1:1 across dates = no drift',
    template='plotly_white', height=640)
fig.update_xaxes(title_text='true tank CH4 (ppm)')
fig.update_yaxes(title_text='corrected (ppm)')
fig.show()

In [ ]:
dates = sorted(WINDOWS_BY_DATE)
drift_rows = []
fig = go.Figure()
for inst in INSTRUMENTS:
    c = CH4_TANK_COEFS[inst]
    if c is None:
        continue
    ys = []
    for d in dates:
        sd = CH4_STATS[d]
        col = f'{inst}_mean'
        if col not in sd.columns:
            ys.append(np.nan); continue
        xt = sd['tank_key'].map(lambda k: TANK.get(k, {}).get('CH4_ppm')).astype(float)
        err = (cal.apply_linear(sd[col].astype(float), c) - xt).values
        err = err[np.isfinite(err)]
        rms = float(np.sqrt(np.mean(err ** 2))) if err.size else np.nan
        ys.append(rms)
        drift_rows.append({'instrument': inst, 'date': d, 'rms_err_ppm': rms})
    fig.add_trace(go.Scatter(x=dates, y=ys, mode='lines+markers', name=inst,
                             line=dict(color=INST_COLORS[inst])))
fig.update_layout(
    title='CH4 drift: RMS error from the single Feb-12 calibration, per tank date (flat & low = no drift)',
    xaxis_title='Tank date', yaxis_title='RMS(corrected − true)  [ppm]',
    template='plotly_white', height=420)
fig.add_annotation(text='Feb-12 = fit-noise floor', showarrow=False,
                   xref='paper', yref='paper', x=0.99, y=0.99, xanchor='right', font=dict(color='gray'))
fig.show()
pd.DataFrame(drift_rows).pivot(index='instrument', columns='date', values='rms_err_ppm').round(3)

### A third CH4 candidate — representative points (tank ladder + binned ambient)

Both candidates above sit at opposite extremes of the same underlying problem: `tank` has
excellent range (0–57 ppm) but only 7 points; `reference` has ~318k points but 99.4% of them
sit below 3 ppm, so an unweighted fit is dominated by ambient and extrapolates poorly to
plume/tank concentrations (the RMS-at-tank-points table above). This candidate tries to get
both: a handful of **representative points spanning the full range**, using real tank data
where ambient can't reach and real (but down-weighted-by-count) ambient data where the tank
can't — instead of either "all 7 tank points" or "all ~318k ambient points."

**Construction** (`cal.bin_paired_values`, new): tank-corrected Picarro (`CH4_TANK_COEFS
['Picarro']` applied to Picarro's own reading) is the x-axis throughout.
- **High end**: the same Feb-12 canonical tank-window means already computed above
  (`CH4_STATS[CAL_DATE_CANONICAL]`) — 7 points, using boundaries `parse_tank_details`
  already parsed and this notebook already trusts. No new transition-detection logic.
- **Low end**: the existing ambient pairs (`CH4_CROSS_PAIRS`, tank-corrected on the
  reference side) aggregated into 20 quantile-based bins via `cal.bin_paired_values` — pure
  binning, no stability heuristic, so a bad edge case can't sneak in the way a rolling-window
  "is this stable" guess could.

**This redoes a change attempted once before that was reverted** (see project memory) after
looking wrong live in Jupyter — that attempt used a rolling-flatness heuristic to decide which
tank-period samples were "settled" before pairing, and was never actually checked against a
rendered continuous timeseries before being called done. This time: no stability heuristic at
all (tank points come straight from already-trusted window boundaries), and the continuous
checks below are run and inspected *before* the comparison table is treated as the verdict.

In [ ]:
def ch4_reppoint_coefs(inst, n_ambient_bins=20, ambient_bin_min_n=20):
    '''Third CH4 candidate: tank-corrected Picarro vs target, fit on representative
    points (Feb-12 canonical tank means + quantile-binned ambient pairs) instead of all
    ~318k raw ambient pairs or just the 7 tank points alone.'''
    picarro_coef = CH4_TANK_COEFS['Picarro']
    paired = CH4_CROSS_PAIRS[inst]
    ref_cal = cal.apply_linear(paired['ref'], picarro_coef)
    edges = np.unique(np.quantile(ref_cal, np.linspace(0, 1, n_ambient_bins + 1)))
    ambient_bins = cal.bin_paired_values(ref_cal.values, paired['target'].values,
                                         bins=edges, min_n=ambient_bin_min_n)

    sd = CH4_STATS[CAL_DATE_CANONICAL]
    pic_cal_tank = cal.apply_linear(sd['Picarro_mean'].astype(float), picarro_coef)
    tgt_tank = sd[f'{inst}_mean'].astype(float)
    tank_points = pd.DataFrame({'x_mean': pic_cal_tank, 'y_mean': tgt_tank, 'n': 1}).dropna()

    rep_points = pd.concat([ambient_bins, tank_points], ignore_index=True)
    sl, ic, r2, n = cal.linreg(rep_points['x_mean'], rep_points['y_mean'])
    coef = {'slope': sl, 'intercept': ic, 'r2': r2, 'n': n}
    return coef, rep_points

CH4_REPPOINT_COEFS, CH4_REPPOINTS = {}, {}
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    CH4_REPPOINT_COEFS[inst], CH4_REPPOINTS[inst] = ch4_reppoint_coefs(inst)
    n_tank = int((CH4_REPPOINTS[inst]['n'] == 1).sum())
    n_amb = len(CH4_REPPOINTS[inst]) - n_tank
    c = CH4_REPPOINT_COEFS[inst]
    print(f'{inst}: {n_tank} tank pts + {n_amb} ambient bins = {c["n"]} representative points  '
          f'slope={c["slope"]:.4f}  intercept={c["intercept"]:+.4f}  R2={c["r2"]:.5f}')

**The representative points themselves** (fit line + 1:1), so a sparse or off bin is
visible before trusting anything downstream — same `plot_calibration_scatter` used
everywhere else in this notebook.

In [ ]:
xg, yg = {}, {}
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    xg[inst] = CH4_REPPOINTS[inst]['x_mean'].values
    yg[inst] = CH4_REPPOINTS[inst]['y_mean'].values
fig = cal.plot_calibration_scatter(xg, yg, CH4_REPPOINT_COEFS, INST_COLORS,
        x_title='Picarro CH4, tank-corrected (ppm)', y_title='Instrument CH4 (ppm)',
        title='CH4 representative-point candidate — tank means + quantile-binned ambient, fit + 1:1')
fig.show()

**Three-way comparison at the 7 canonical tank points** — same yardstick as the
`tank`-vs-`reference` table above, now with `reference_binned` alongside.

In [ ]:
three_way_rows = []
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    candidates = {'tank': CH4_TANK_COEFS.get(inst), 'reference': CH4_CROSS_COEFS.get(inst),
                  'reference_binned': CH4_REPPOINT_COEFS.get(inst)}
    cmp = cal.compare_candidate_coefs(candidates, CH4_STATS[CAL_DATE_CANONICAL], TANK, 'CH4_ppm', inst)
    cmp.insert(0, 'instrument', inst)
    three_way_rows.append(cmp)
CH4_THREE_WAY = pd.concat(three_way_rows)
CH4_THREE_WAY.round(4)

**And at ambient concentrations** — RMS against tank-corrected Picarro over the full
ambient pairing population (a different, much larger population than the 7 tank points
above; not apples-to-apples with the table, same caveat as the `reference` candidate's own
ambient-population aside earlier). This is the number that should stay closer to
`reference`'s tight ambient fit than to `tank`'s, if the hybrid is doing its job.

In [ ]:
ambient_rms_rows = []
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    paired = CH4_CROSS_PAIRS[inst]
    ref_cal = cal.apply_linear(paired['ref'], CH4_TANK_COEFS['Picarro'])
    for name, coef in [('tank', CH4_TANK_COEFS[inst]), ('reference', CH4_CROSS_COEFS[inst]),
                       ('reference_binned', CH4_REPPOINT_COEFS[inst])]:
        corrected = cal.apply_linear(paired['target'], coef)
        resid = corrected - ref_cal
        ambient_rms_rows.append({'instrument': inst, 'candidate': name,
                                 'ambient_population_RMS_ppm': float(np.sqrt(np.mean(resid ** 2)))})
pd.DataFrame(ambient_rms_rows).pivot(index='instrument', columns='candidate',
        values='ambient_population_RMS_ppm').round(4)

**Mandatory before trusting any of the above**: render the actual continuous
corrected-vs-Picarro timeseries, at two windows — the same ambient `ZOOM_START`/`ZOOM_END`
used everywhere else in this notebook, and a window spanning the **entire Feb-12 tank
ladder** (the regime the previous, reverted attempt was never actually checked against).
Reference = Picarro, tank-corrected.

In [ ]:
CH4_PICARRO_CAL = cal.apply_linear(CH4['Picarro'], CH4_TANK_COEFS['Picarro'])
cal_ch4_reppoint = {inst: cal.apply_linear(CH4[inst], c) for inst, c in CH4_REPPOINT_COEFS.items()}
_cmp_colors = {'Picarro (tank-corrected, reference)': INST_COLORS['Picarro'],
               'Ultra460': INST_COLORS['Ultra460'], 'Ultra321': INST_COLORS['Ultra321'],
               'Pico017': INST_COLORS['Pico017']}
fig = cal.plot_raw_corrected_vs_reference(
    CH4_PICARRO_CAL, 'Picarro (tank-corrected, reference)',
    {inst: (CH4[inst], cal_ch4_reppoint[inst]) for inst in CH4_REPPOINT_COEFS},
    _cmp_colors, ZOOM_START, ZOOM_END,
    title=f'CH4 representative-point candidate — AMBIENT window — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC',
    y_title='CH4 (ppm)')
fig.show()

LADDER_START = pd.Timestamp('2026-02-12 22:20', tz='UTC')
LADDER_END   = pd.Timestamp('2026-02-12 23:15', tz='UTC')
fig2 = cal.plot_raw_corrected_vs_reference(
    CH4_PICARRO_CAL, 'Picarro (tank-corrected, reference)',
    {inst: (CH4[inst], cal_ch4_reppoint[inst]) for inst in CH4_REPPOINT_COEFS},
    _cmp_colors, LADDER_START, LADDER_END,
    title=f'CH4 representative-point candidate — FEB-12 TANK LADDER — {LADDER_START:%H:%M}–{LADDER_END:%H:%M} UTC',
    y_title='CH4 (ppm)')
fig2.show()

**Not locked in.** This candidate is not written to `CAL_METHOD_LOCKED` in
`04_calibration.ipynb` — it's presented here as a real, verified third option (numbers and
continuous plots both checked, unlike the previous attempt) for a human decision, same as
every other candidate this notebook computes.

---
## D — C3H8: no second candidate

Only Ultra321 measures C3H8, so there's no other instrument to cross-calibrate against
— `reference` isn't a candidate here at all, not a comparison that was run and lost.
Tank-anchored (the official notebook's Section C) is the only viable method, forced, not
a choice. Nothing to compare in this section.

---
## E — C2H6: why `tank` isn't viable, and the Ultra321 interference evidence

**Why the tank method is unusable for C2H6**: the ambient plume distribution sits far
above the single certified point. `cal.assess_tank_coverage` makes this a number instead
of an eyeballed plot: the tank's max certified C2H6 point against real ambient/plume
percentiles.

In [ ]:
CAL_WINDOW_PAD_MIN = 5
U460_AMB = cal.restrict_series(C2H6['Ultra460'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
PICO_AMB = cal.restrict_series(C2H6['Pico017'],  WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
U321_AMB = cal.restrict_series(C2H6['Ultra321'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
CH4_PIC_AMB   = cal.restrict_series(CH4['Picarro'],   WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)
C3H8_U321_AMB = cal.restrict_series(C3H8['Ultra321'], WYO_DATES, WINDOWS_BY_DATE, pad_min=CAL_WINDOW_PAD_MIN)

coverage = cal.assess_tank_coverage(TANK, 'C2H6_ppb', U460_AMB, quantiles=(0.5, 0.99, 1.0))
coverage.round(3)

In [ ]:
u460 = U460_AMB.dropna()
levels = [
    ('NOAA tank — only certified point', TANK['NOAA']['C2H6_ppb'], 'black'),
    ('ambient median',                    float(u460.median()),      '#8a8a8a'),
    ('ambient 99th percentile',           float(u460.quantile(0.99)),'#8a8a8a'),
    ('plume-peak fit threshold (50 ppb)', 50.0,                      'gray'),
    ('largest plume peak',                float(u460.max()),         INST_COLORS['Ultra460']),
]
fig = cal.plot_range_levels(levels, x_title='C2H6 (ppb, log scale)',
        title='Why C2H6 cannot be tank-calibrated: one tank point vs the range we must calibrate')
fig.show()

`find_peak_matches` locates plume peaks in the Ultra460 reference (per day) and, at
each peak time, grabs the local max of every other series within a ±10 s window — Pico017
and Ultra321 as calibration targets, plus Picarro CH4 and Ultra321 C3H8 as interference
diagnostics.

In [ ]:
PEAK_HEIGHT_PPB     = 50.0
PEAK_PROMINENCE_PPB = 15.0
PEAK_MIN_DISTANCE_S = 30
PEAK_MATCH_WINDOW_S = 10

peaks_df = cal.find_peak_matches(
    U460_AMB,
    {'Pico017': PICO_AMB, 'Ultra321': U321_AMB,
     'CH4_picarro': CH4_PIC_AMB, 'C3H8_ultra321': C3H8_U321_AMB},
    height=PEAK_HEIGHT_PPB, prominence=PEAK_PROMINENCE_PPB,
    min_distance_s=PEAK_MIN_DISTANCE_S, window_s=PEAK_MATCH_WINDOW_S)
print(f'Plume peaks found: {len(peaks_df)}')
peaks_df.groupby('date').size().rename('n_peaks').to_frame()

**See what the peak finder did — and meet the Ultra321 problem.** `EXAMPLE_DATE`
below (a clean ambient co-deployment day, not a tank-cal day). Ultra460 (reference) and
Pico017 share a scale and their plume peaks line up — good. **Ultra321 is plotted on its
own axis below**, because it can't share one with the others: its C2H6 baseline sits near
**−135 ppb** and it reads negative most of the campaign. That broken baseline — on top
of the C3H8 cross-talk shown next — is the root of its failed fit.

In [ ]:
EXAMPLE_DATE = '2026-02-11'   # any WYO ambient co-deployment day (not tank-cal); Feb 11 is the busiest
peak_times = peaks_df.loc[peaks_df['date'] == EXAMPLE_DATE, 'peak_time']

fig = cal.plot_peaks_highlighted(
    {'Ultra460': U460_AMB[EXAMPLE_DATE], 'Pico017': PICO_AMB[EXAMPLE_DATE]}, peak_times, INST_COLORS,
    title=f'C2H6 plume peaks on {EXAMPLE_DATE} — Ultra460 (ref) & Pico017 (x = matched peaks)',
    y_title='C2H6 (ppb)')
fig.show()

fig = cal.plot_peaks_highlighted(
    {'Ultra321': U321_AMB[EXAMPLE_DATE]}, peak_times, INST_COLORS,
    title=f'Ultra321 C2H6 on {EXAMPLE_DATE} — SEPARATE AXIS: broken (negative) baseline near -135 ppb',
    y_title='C2H6 (ppb)')
fig.show()

**Re-fit the (only viable) reference candidate**, to get the residuals the
interference diagnostic below needs. Deterministically identical to what the official
notebook computes in its Section D.

In [ ]:
Z_REF, Z_REF_SPREAD, _ = cal.ambient_baseline_stats(U460_AMB, q=0.5)
C2H6_COEFS = {}
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    sub = peaks_df.dropna(subset=['ref', inst])
    z_tgt, z_tgt_spread, _ = cal.ambient_baseline_stats(amb, q=0.5)
    C2H6_COEFS[inst] = cal.fit_reference_cal(sub['ref'].values, sub[inst].values,
                                             anchor='pinned', z_ref=Z_REF, z_tgt=z_tgt)

# Numeric candidate-comparison table anyway, for consistency with CH4/C3H8 above, even
# though there's only ever one candidate to show (spot-checked at the tank's 2 usable
# points: N2_zero and NOAA — everything else is NaN for C2H6 and gets dropped).
canonical_stats_c2h6 = cal.window_stats(C2H6, WINDOWS_BY_DATE[CAL_DATE_CANONICAL])
for inst in ['Pico017', 'Ultra321']:
    cmp = cal.compare_candidate_coefs({'reference': C2H6_COEFS[inst]}, canonical_stats_c2h6, TANK, 'C2H6_ppb', inst)
    cmp.insert(0, 'instrument', inst)
    print(cmp.round(3).to_string())

**The interference diagnostic.** If a fit residual trends with CH4 or C3H8 level,
that channel is contaminating the C2H6 retrieval. Pico017 should scatter flat around
zero; Ultra321 vs C3H8 is the smoking gun.

In [ ]:
for inst in ['Pico017', 'Ultra321']:
    c = C2H6_COEFS[inst]
    sub = peaks_df.dropna(subset=['ref', inst])
    resid = sub[inst].values - (c['slope'] * sub['ref'].values + c['intercept'])
    for diag_col in ['CH4_picarro', 'C3H8_ultra321']:
        fig, corr = cal.plot_residual_diagnostic(
            resid, sub[diag_col].values, f'{diag_col} at peak', color=INST_COLORS[inst])
        fig.update_layout(title=f'{inst}: ' + fig.layout.title.text)
        fig.show()

**Raw vs corrected on the example day**, Pico017 (good fit) next to Ultra321 (poor
fit), so the quality gap is visible in the actual signal, not just in a scatter R².

In [ ]:
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    raw = amb[EXAMPLE_DATE]
    corrected = cal.apply_linear(raw, C2H6_COEFS[inst])
    fig = cal.plot_raw_vs_corrected(raw, corrected,
            f'{inst} C2H6 raw vs corrected — {EXAMPLE_DATE}', 'C2H6 (ppb)')
    fig.show()

### A second C2H6 candidate — binned ambient + peaks, unpinned

Same idea as CH4's `reference_binned` candidate — pick representative points spanning the
range instead of either "one anchor point" or "raw noisy pairs" — but **there are no tank
values to fold in for C2H6** (that's the whole reason `tank` isn't viable here at all, per
`assess_tank_coverage` above). So both ends of this candidate come from real ambient/peak
data instead:

- **Low end**: pair Ultra460 against each target at every ambient (tank-excluded) timestamp
  (`pair_series_nearest`, same primitive as CH4's cross-cal) — restricted to `ref < 50 ppb`
  so it doesn't overlap the peak-detection threshold below, then aggregated into 10
  quantile bins (`cal.bin_paired_values`) instead of the current candidate's single
  ambient-median anchor point.
- **High end**: the same matched peaks (`peaks_df`) already computed above.
- **Fit**: plain unpinned `linreg` on (ambient bins + peaks) — no forced anchor, since the
  ambient bins now carry real information about the low-end relationship instead of just
  one point.

Raw (unbinned) ambient correlation is much weaker here than CH4's (r=0.54 Pico017, r=0.26
Ultra321 at 1-second pairing — C2H6's ambient signal sits much closer to each instrument's
noise floor than CH4's does), which is exactly why the *current* candidate uses only peaks
+ one pinned anchor rather than raw ambient pairs. Binning is what makes the ambient
information usable at all.

In [ ]:
def c2h6_reppoint_coefs(inst, amb_series, n_ambient_bins=10, ambient_bin_min_n=1000, ambient_max_ppb=50.0):
    '''Second C2H6 candidate: quantile-binned sub-peak-threshold ambient pairs + matched
    peaks, fit unpinned -- vs the current candidate's single ambient-median anchor + peaks.'''
    paired = cal.pair_series_nearest(U460_AMB, amb_series, tolerance_s=1)
    sub = paired[paired['ref'] < ambient_max_ppb]
    edges = np.unique(np.quantile(sub['ref'], np.linspace(0, 1, n_ambient_bins + 1)))
    ambient_bins = cal.bin_paired_values(sub['ref'].values, sub['target'].values,
                                         bins=edges, min_n=ambient_bin_min_n)

    psub = peaks_df.dropna(subset=['ref', inst])
    peak_points = pd.DataFrame({'x_mean': psub['ref'].values, 'y_mean': psub[inst].values, 'n': 1})

    rep_points = pd.concat([ambient_bins, peak_points], ignore_index=True)
    sl, ic, r2, n = cal.linreg(rep_points['x_mean'], rep_points['y_mean'])
    coef = {'slope': sl, 'intercept': ic, 'r2': r2, 'n': n}
    return coef, rep_points, paired

C2H6_REPPOINT_COEFS, C2H6_REPPOINTS, C2H6_AMB_PAIRED = {}, {}, {}
for inst, amb in [('Pico017', PICO_AMB), ('Ultra321', U321_AMB)]:
    coef, rep, paired = c2h6_reppoint_coefs(inst, amb)
    C2H6_REPPOINT_COEFS[inst], C2H6_REPPOINTS[inst], C2H6_AMB_PAIRED[inst] = coef, rep, paired
    n_peaks = int((rep['n'] == 1).sum())
    n_amb = len(rep) - n_peaks
    print(f'{inst}: {n_amb} ambient bins + {n_peaks} peaks = {coef["n"]} representative points  '
          f'slope={coef["slope"]:.4f}  intercept={coef["intercept"]:+.4f}  R2={coef["r2"]:.5f}')

**The representative points themselves** — same sanity check as CH4's, so a bad or
sparse bin is visible before trusting anything downstream.

In [ ]:
xg, yg = {}, {}
for inst in ['Pico017', 'Ultra321']:
    xg[inst] = C2H6_REPPOINTS[inst]['x_mean'].values
    yg[inst] = C2H6_REPPOINTS[inst]['y_mean'].values
fig = cal.plot_calibration_scatter(xg, yg, C2H6_REPPOINT_COEFS, INST_COLORS,
        x_title='Ultra460 C2H6 (ppb)', y_title='Instrument C2H6 (ppb)',
        title='C2H6 second candidate — quantile-binned ambient + peaks, fit + 1:1')
fig.show()

**Comparison**: the current pinned zero+span candidate vs. this binned-unpinned one,
on two yardsticks — the tank's 2 usable points (spot check only, same caveat as before:
N2_zero and NOAA, not real evidence either way) and RMS against Ultra460 over the *entire*
ambient population (the more meaningful number here, since neither candidate was tank-fit).

In [ ]:
c2h6_cmp_rows = []
for inst in ['Pico017', 'Ultra321']:
    candidates = {'reference': C2H6_COEFS[inst], 'reference_binned': C2H6_REPPOINT_COEFS[inst]}
    cmp = cal.compare_candidate_coefs(candidates, canonical_stats_c2h6, TANK,
            'C2H6_ppb', inst)
    cmp.insert(0, 'instrument', inst)
    c2h6_cmp_rows.append(cmp)
print('At the tank\'s 2 usable points (spot check only):')
print(pd.concat(c2h6_cmp_rows).round(3).to_string())
print()

amb_rows = []
for inst in ['Pico017', 'Ultra321']:
    paired = C2H6_AMB_PAIRED[inst]
    for name, coef in [('reference', C2H6_COEFS[inst]), ('reference_binned', C2H6_REPPOINT_COEFS[inst])]:
        corrected = cal.apply_linear(paired['target'], coef)
        resid = corrected - paired['ref']
        amb_rows.append({'instrument': inst, 'candidate': name,
                         'ambient_population_RMS_ppb': float(np.sqrt(np.mean(resid ** 2)))})
print('Over the full ambient population (vs Ultra460):')
pd.DataFrame(amb_rows).pivot(index='instrument', columns='candidate', values='ambient_population_RMS_ppb').round(3)

**Mandatory before trusting any of the above**: render the actual continuous
corrected-vs-Ultra460 timeseries — the same ambient `ZOOM_START`/`ZOOM_END` window used
throughout this notebook, and `EXAMPLE_DATE`'s densest plume cluster (same center used for
the interference diagnostic above), for both candidates side by side.

In [ ]:
cal_c2h6_reppoint = {inst: cal.apply_linear(C2H6[inst], c) for inst, c in C2H6_REPPOINT_COEFS.items()}
_cmp_colors = {'Ultra460 (reference)': INST_COLORS['Ultra460'],
               'Pico017': INST_COLORS['Pico017'], 'Ultra321': INST_COLORS['Ultra321']}
fig = cal.plot_raw_corrected_vs_reference(
    C2H6['Ultra460'], 'Ultra460 (reference)',
    {inst: (C2H6[inst], cal_c2h6_reppoint[inst]) for inst in C2H6_REPPOINT_COEFS},
    _cmp_colors, ZOOM_START, ZOOM_END,
    title=f'C2H6 second candidate — AMBIENT window — {ZOOM_START:%Y-%m-%d %H:%M}–{ZOOM_END:%H:%M} UTC',
    y_title='C2H6 (ppb)')
fig.show()

_pt = peaks_df.loc[peaks_df['date'] == EXAMPLE_DATE, 'peak_time'].sort_values()
_center = max(_pt, key=lambda t: ((_pt >= t - pd.Timedelta('15min')) & (_pt <= t + pd.Timedelta('15min'))).sum())
PLUME_WIN_START = _center - pd.Timedelta('15min')
PLUME_WIN_END   = _center + pd.Timedelta('15min')
fig2 = cal.plot_raw_corrected_vs_reference(
    C2H6['Ultra460'], 'Ultra460 (reference)',
    {inst: (C2H6[inst], cal_c2h6_reppoint[inst]) for inst in C2H6_REPPOINT_COEFS},
    _cmp_colors, PLUME_WIN_START, PLUME_WIN_END,
    title=f'C2H6 second candidate — densest plume cluster, {EXAMPLE_DATE} — {PLUME_WIN_START:%H:%M}–{PLUME_WIN_END:%H:%M} UTC',
    y_title='C2H6 (ppb)')
fig2.show()

**Honest result — not a uniform win like CH4's.**
- **Pico017**: essentially a wash. Coefficients land close to the current candidate
  (slope 0.88 vs 0.88, intercept +14.8 vs +15.5) and ambient-population RMS is marginally
  better. The binned ambient data confirms Pico017 tracks Ultra460 reasonably well even
  below the peak-detection threshold, not just at peaks.
- **Ultra321**: **worse**, not better, by ambient-population RMS — and this is itself a
  useful, cleaner finding, not a failure of the method. Binned by Ultra460 level, Ultra321's
  ambient response is nearly flat (its bin means span roughly −140 to −127 ppb across the
  *entire* 4–17 ppb ambient range of Ultra460) — i.e. **Ultra321 barely responds to real
  ambient C2H6 variation at all**, a more direct demonstration of the channel's brokenness
  than the peak-only R²≈0.83 already established. Pinning through the ambient median (the
  current candidate) is the better practical choice for Ultra321 specifically, because
  letting an unpinned fit chase a near-flat, noise-dominated ambient relationship pulls the
  fit away from the one anchor point that's actually reliable.

**Not locked in, same as CH4's third candidate.** For Pico017 this is a viable, marginal
alternative to what's applied; for Ultra321 it's additional diagnostic evidence for the
existing drop recommendation, not a proposed replacement.

---
## Decision recap

- **CH4 → `tank`.** Both candidates viable; tank RMS-at-tank-points (0.08–0.27 ppm) beats
  the Picarro cross-cal's out-of-sample RMS at the same points (0.19–0.50 ppm), and this
  is a plume-detection campaign — accuracy at plume concentrations matters more than the
  cross-cal's tighter ambient-only agreement. See Section C's `CH4_COMPARISON` table.
- **C3H8 → `tank`.** Forced: only Ultra321 measures it, no reference candidate exists
  (Section D).
- **C2H6 → `reference`.** Forced: the tank has one certified point (NOAA, 1.63 ppb),
  demonstrably far below the ambient/plume range that needs calibrating (Section E's
  `assess_tank_coverage`). Ultra460 stands in as the reference. Ultra321's fit is
  additionally flagged low-confidence: R²(span) ≈ 0.83 and its residual correlates
  ≈ +0.95 with C3H8 (Section E's interference diagnostic) — a spectral cross-talk no
  anchor choice can fix; leaning is to drop it.

These are the choices encoded in `04_calibration.ipynb`'s `CAL_METHOD_LOCKED`. Re-run
this notebook and update that dict (with a matching Section C/D code change there) if
new tank or ambient data should prompt revisiting a lock.